# Análisis Global de Mercado: Obra Nueva vs Segunda Mano

Esta notebook consolida los datos de todos los scrapeos locales en `output/`, aplica los filtros estándar, y realiza análisis estadísticos y visuales sobre el precio por metro cuadrado (Price/SqM) para responder a las tres grandes preguntas analíticas.

In [1]:
if 'graficos_base64' not in globals():
    graficos_base64 = {}
import io
import base64
import os
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import webbrowser
import time

# Configuración visual
sns.set_theme(style="whitegrid")
# 1. Consolidación de Datos
output_dir = '../output'
all_data = []
print_lines = []

for folder in os.listdir(output_dir):
    folder_path = os.path.join(output_dir, folder)
    if not os.path.isdir(folder_path):
        continue
    # Parsear nombre de la carpeta (Ej: 110000-320000_buy_new_Alic_afueras)
    parts = folder.split('_')
    if len(parts) >= 5 and parts[1].lower() == 'buy':
        p_type = 'New' if 'new' in parts[2].lower() else 'Used'
        city = parts[3].capitalize()
        if city == 'Alic': city = 'Ali'  # Normalizar nombre
        zone = parts[4].capitalize()
   
        # Encontrar CSV
        csvs = glob.glob(os.path.join(folder_path, '*.csv'))
        if not csvs: continue
            
        df = pd.read_csv(csvs[0])
        # --- LIMPIEZA DE COLUMNAS ---
        if 'Price (€)' in df.columns:
            if df['Price (€)'].dtype == 'object':
                df['Price (€)'] = pd.to_numeric(df['Price (€)'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).str.replace('€', '', regex=False).str.strip(), errors='coerce')
        if 'Area (m²)' in df.columns:
            if df['Area (m²)'].dtype == 'object':
                df['Area (m²)'] = pd.to_numeric(df['Area (m²)'].astype(str).str.replace('m²', '', regex=False).str.replace('.', '', regex=False).str.strip(), errors='coerce')
        if 'Hab' in df.columns:
            if df['Hab'].dtype == 'object':
                df['Hab'] = pd.to_numeric(df['Hab'].astype(str).str.replace('hab.', '', regex=False).str.replace('-', '', regex=False).str.strip(), errors='coerce')
        
        # Normalizar zona a 'Dentro' o 'Afueras'
        norm_zone = 'Dentro' if 'dentro' in zone.lower() else 'Afueras'
        
        initial_len = len(df)
        
        # Filtros Estrictos (Nulos, Precio > 320k, Area < 65)
        df = df.dropna()
        if 'Price (€)' in df.columns: df = df[df['Price (€)'] <= 320000]
        if 'Area (m²)' in df.columns: df = df[df['Area (m²)'] >= 65]
        
        final_len = len(df)
        type_key = 0 if p_type == 'Used' else 1
        print_lines.append((type_key, city, norm_zone, f"[{p_type:4} | {city:3} | {norm_zone:7}] Originales: {initial_len:4} | Filtradas: {final_len:4}"))
        
        if 'Price (€)' in df.columns and 'Area (m²)' in df.columns and len(df) > 0:
            df['Price/SqM'] = df['Price (€)'] / df['Area (m²)']
            df['Type'] = p_type
            df['City'] = city
            # Normalizar zona a 'Dentro' o 'Afueras'

            df['Zone'] = norm_zone
            df['Location_Full'] = f"{city} {norm_zone}"
            
            all_data.append(df)


# Imprimir reporte de filas ordenado
print_lines.sort(key=lambda x: (x[0], x[1], x[2]))
for line in print_lines:
    print(line[3])

df_master = pd.concat(all_data, ignore_index=True)
df_master['Price/SqM'] = df_master['Price/SqM'].astype(float)
print(f"Base de datos consolidada con {len(df_master)} propiedades limpias.")



[Used | Ali | Afueras] Originales:  157 | Filtradas:  157
[Used | Ali | Dentro ] Originales:  171 | Filtradas:  171
[Used | Bcn | Afueras] Originales:  143 | Filtradas:  143
[Used | Bcn | Dentro ] Originales:  131 | Filtradas:  131
[Used | Mal | Afueras] Originales:  124 | Filtradas:  124
[Used | Mal | Dentro ] Originales:  147 | Filtradas:  147
[Used | Sev | Afueras] Originales:  163 | Filtradas:  163
[Used | Sev | Dentro ] Originales:  150 | Filtradas:  150
[Used | Val | Afueras] Originales:  139 | Filtradas:  139
[Used | Val | Dentro ] Originales:  150 | Filtradas:  150
[New  | Ali | Afueras] Originales:  134 | Filtradas:   90
[New  | Ali | Dentro ] Originales:   63 | Filtradas:   36
[New  | Bcn | Afueras] Originales:  139 | Filtradas:   37
[New  | Bcn | Dentro ] Originales:  129 | Filtradas:   19
[New  | Costa | Afueras] Originales:   46 | Filtradas:   36
[New  | Costa | Afueras] Originales:   48 | Filtradas:   15
[New  | Costa | Afueras] Originales:   68 | Filtradas:   30
[New  | 

In [2]:
print("\n--- head (df_master) ---")
display(df_master.head())
print("\n--- Descriptive Stats (df_master) ---")
print(df_master.describe().round(1))
print("\n--- TIPOS DE DATOS (df_master) ---")
print(df_master.dtypes)



--- head (df_master) ---


,Link,Price (€),Area (m²),Hab,Location,Price/SqM,Type,City,Zone,Location_Full
0,https://www.idealista.com/inmueble/108338354/,135000,90.0,3.0,San Vicente del Raspeig,1500.000000,Used,Ali,Afueras,Ali Afueras
1,https://www.idealista.com/inmueble/111244240/,139000,86.0,3.0,San Vicente del Raspeig,1616.279070,Used,Ali,Afueras,Ali Afueras
2,https://www.idealista.com/inmueble/110083642/,142000,77.0,2.0,"Centro, San Juan de Alicante",1844.155844,Used,Ali,Afueras,Ali Afueras
3,https://www.idealista.com/inmueble/110757639/,143000,85.0,2.0,"Torrellano, Elche / Elx",1682.352941,Used,Ali,Afueras,Ali Afueras
4,https://www.idealista.com/inmueble/110950619/,149500,72.0,2.0,"Centro, San Vicente del Raspeig",2076.388889,Used,Ali,Afueras,Ali Afueras



--- Descriptive Stats (df_master) ---
       Price (€)  Area (m²)     Hab  Price/SqM
count     2607.0     2607.0  2607.0     2607.0
mean    219908.9       93.4     2.7     2438.2
std      52031.3       18.9     0.7      739.4
min     103300.0       65.0     1.0      884.9
25%     173000.0       80.0     2.0     1859.4
50%     220400.0       90.0     3.0     2362.5
75%     260000.0      101.0     3.0     2920.3
max     320000.0      263.0     5.0     4833.3

--- TIPOS DE DATOS (df_master) ---
Link              object
Price (€)          int64
Area (m²)        float64
Hab              float64
Location          object
Price/SqM        float64
Type              object
City              object
Zone              object
Location_Full     object
dtype: object


## Pregunta 1: Distribución entre Ciudades por Segmento Específico
Comparamos la dispersión de precios para una misma categoría en distintas ciudades.

In [ ]:
plt.figure(figsize=(16, 10))
plt.suptitle('Distribución de Precios/m² por Ciudad (Segmentado)', fontsize=18, y=1.02)

combinations = [('Used', 'Dentro'), ('Used', 'Afueras'), ('New', 'Dentro'), ('New', 'Afueras')]
order = ['Ali', 'Bcn', 'Mal', 'Sev', 'Val']

for i, (t, z) in enumerate(combinations, 1):
    plt.subplot(2, 2, i)
    subset = df_master[(df_master['Type'] == t) & (df_master['Zone'] == z)]
    
    sns.boxplot(data=subset, x='City', y='Price/SqM', order=order, palette='Set2')
    plt.title(f'{t} - {z}', fontsize=14)
    plt.xlabel('')
    plt.ylabel('€/m²')

plt.tight_layout()
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
buf.seek(0)
graficos_base64['graph1'] = base64.b64encode(buf.read()).decode('utf-8')
buf.close()
plt.show()

## Pregunta 2: Dinámica Interna de Cada Ciudad
Analizamos cómo varían las medianas de precio entre Used vs New y Dentro vs Afueras para una ciudad en concreto.

In [ ]:
medians = df_master.groupby(['City', 'Type', 'Zone'])['Price/SqM'].median().reset_index()
medians['Category'] = medians['Type'] + " " + medians['Zone']
category_order = ['Used Dentro', 'Used Afueras', 'New Dentro', 'New Afueras']

plt.figure(figsize=(16, 8))
ax = sns.barplot(data=medians, x='City', y='Price/SqM', hue='Category', 
                 hue_order=category_order, order=order, palette='muted')

plt.title('Dinámica Interna por Ciudad: Medianas de Precio/m²', fontsize=18)
plt.ylabel('Precio Mediano (€/m²)')
plt.xlabel('Ciudad')

# Añadir valores encima de las barras
for p in ax.patches:
    height = p.get_height()
    if pd.notnull(height) and height > 0:
        ax.text(p.get_x() + p.get_width()/2., height + 20, f'€{int(height)}', 
                ha="center", va="bottom", fontsize=10, rotation=45)

plt.legend(title='Segmento')
plt.tight_layout()
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
buf.seek(0)
graficos_base64['graph2'] = base64.b64encode(buf.read()).decode('utf-8')
buf.close()
plt.show()

# --- CÁLCULO DE VARIACIONES PORCENTUALES ---
pivot = medians.pivot_table(index='City', columns=['Type', 'Zone'], values='Price/SqM')

# Variación de Afueras a Centro
used_out_in = ((pivot['Used']['Dentro'] - pivot['Used']['Afueras']) / pivot['Used']['Afueras'] * 100).mean()
new_out_in = ((pivot['New']['Dentro'] - pivot['New']['Afueras']) / pivot['New']['Afueras'] * 100).mean()
avg_out_in = (used_out_in + new_out_in) / 2

# Variación de Usado a Nuevo
in_used_new = ((pivot['New']['Dentro'] - pivot['Used']['Dentro']) / pivot['Used']['Dentro'] * 100).mean()
out_used_new = ((pivot['New']['Afueras'] - pivot['Used']['Afueras']) / pivot['Used']['Afueras'] * 100).mean()
avg_used_new = (in_used_new + out_used_new) / 2

print("\n=======================================================")
print("  VARIACIONES PORCENTUALES PROMEDIO (Entre las 5 Ciudades)")
print("=======================================================")
print(f"🔹 DE AFUERAS AL CENTRO:")
print(f"   - En pisos Usados aumenta un: +{used_out_in:.0f}%")
print(f"   - En pisos Nuevos aumenta un: +{new_out_in:.0f}%")
print(f"   ► PROMEDIO GENERAL Centro vs Afueras: +{avg_out_in:.0f}%")
print("")
print(f"🔹 DE USADO A OBRA NUEVA:")
print(f"   - En las Afueras aumenta un: +{out_used_new:.0f}%")
print(f"   - En el Centro aumenta un: +{in_used_new:.0f}%")
print(f"   ► PROMEDIO GENERAL Nuevo vs Usado: +{avg_used_new:.0f}%")


## Pregunta 3: Visión General del Mercado (Macro)
Estadísticas descriptivas a nivel global (toda España) para New vs Used, y Dentro vs Afueras.

In [ ]:
plt.figure(figsize=(16, 6))

# KDE New vs Used
plt.subplot(1, 2, 1)
sns.kdeplot(data=df_master, x='Price/SqM', hue='Type', fill=True, common_norm=False, palette='coolwarm')
plt.title('Global: New vs Used (Densidad)', fontsize=14)
plt.xlabel('€/m²')

# KDE Dentro vs Afueras
plt.subplot(1, 2, 2)
sns.kdeplot(data=df_master, x='Price/SqM', hue='Zone', fill=True, common_norm=False, palette='viridis')
plt.title('Global: Dentro vs Afueras (Densidad)', fontsize=14)
plt.xlabel('€/m²')

plt.tight_layout()
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
buf.seek(0)
graficos_base64['graph3'] = base64.b64encode(buf.read()).decode('utf-8')
buf.close()
plt.show()

print("==============================================")
print("    ESTADÍSTICAS MACRO: NEW vs USED (Global) ")
print("==============================================")
display(df_master.groupby('Type')['Price/SqM'].describe().round(0))

print("\n==============================================")
print(" ESTADÍSTICAS MACRO: DENTRO vs AFUERAS (Global) ")
print("==============================================")
display(df_master.groupby('Zone')['Price/SqM'].describe().round(0))

## Pregunta 4: Densidad de los 4 Grupos
Comparación general superponiendo los 4 grupos.

In [ ]:
plt.figure(figsize=(12, 6))
df_master['Segment'] = df_master['Type'] + " " + df_master['Zone']

# Asignar colores fijos a cada segmento para que coincidan las curvas y las líneas
segments = sorted(df_master['Segment'].dropna().unique())
palette = sns.color_palette('tab10', len(segments))
color_dict = dict(zip(segments, palette))

ax = sns.kdeplot(data=df_master, x='Price/SqM', hue='Segment', fill=True, common_norm=False, palette=color_dict)

# Añadir líneas verticales para las medianas
for seg in segments:
    median_val = df_master[df_master['Segment'] == seg]['Price/SqM'].median()
    ax.axvline(median_val, color=color_dict[seg], linestyle='--', linewidth=2, alpha=0.8)
    # Opcional: Escribir el texto de la mediana junto a la línea
    # ax.text(median_val + 10, ax.get_ylim()[1]*0.9, f'€{int(median_val)}', color=color_dict[seg], rotation=90)

plt.title('Global: Densidad de los 4 Grupos Principales (con Medianas)', fontsize=16)
plt.xlabel('Precio (€/m²)')
plt.tight_layout()
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
buf.seek(0)
graficos_base64['graph4'] = base64.b64encode(buf.read()).decode('utf-8')
buf.close()
plt.show()


## Generador de Reporte HTML Independiente
Guarda todos los gráficos generados en un único archivo HTML listo para compartir.

In [ ]:
# --- CONTEO POR PROMOCIÓN (OBRA NUEVA) ---
if 'Link' in propiedades_finales.columns:
    print("\n--- Unidades disponibles por Promoción ---")
    # Recorta el link hasta el ID del proyecto (antes de /inmueble/)
    promo_links = propiedades_finales['Link'].dropna().apply(lambda x: x.split('/inmueble/')[0] + '/' if '/inmueble/' in x else x)
    conteo = promo_links.value_counts()
    print(conteo)
    print("\n")

import base64
import webbrowser

    <style>
        body { font-family: Arial, sans-serif; background-color: #f4f4f9; color: #333; padding: 20px; }
        h1 { text-align: center; color: #2c3e50; }
        .container { display: flex; flex-direction: column; gap: 40px; margin-top: 30px; }
        .card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); text-align: center; }
        .card img { max-width: 100%; height: auto; border-radius: 4px; }
    </style>
"""

html = f"<!DOCTYPE html><html lang='es'><head><meta charset='UTF-8'><title>Reporte Global</title>{css_styles}</head><body>"
html += "<h1>Análisis Global del Mercado Inmobiliario</h1><div class='container'>"

sections = ["1. Distribución por Ciudad", "2. Dinámica Interna", "3. Macro (New vs Used / Dentro vs Afueras)", "4. Macro (4 Grupos)"]
for i, title in enumerate(sections, 1):
    key = f"graph{i}"
    if key in graficos_base64:
        html += f"<div class='card'><h2>{title}</h2><img src='data:image/png;base64,{graficos_base64[key]}' alt='Graph {i}'></div>\n"

html += "</div></body></html>"
report_filename = "reporte_global_independiente.html"
with open(report_filename, "w", encoding="utf-8") as f:
    f.write(html)


print(f"¡Reporte HTML guardado en {report_filename}!")
webbrowser.open('file://' + os.path.realpath(report_filename))


## 5. Selección y Filtrado Personalizado
Aquí puedes reducir la tabla maestra a los grupos de ciudades y tipos que te interesen evaluar a nivel de propiedad individual.

In [ ]:
# Conteo de grupos
conteo_grupos = df_master.groupby(['Type', 'City', 'Zone']).size().reset_index(name='Count')
display(conteo_grupos)

In [29]:
# MODO DE FILTRADO:
# Opcion 1: 'Ciudades_Nuevas' (Filtra propiedades New en Bcn, Ali, Val)
# Opcion 2: 'Costa' (Filtra las búsquedas personalizadas que contienen 'Costa')

modo = 'Costa' # <-- Cambia esto a 'Nuevas' o 'Costa'

if modo == 'Nuevas':
    mi_seleccion = df_master[
        (df_master['Type'] == 'New') & 
        (df_master['City'].isin(['Bcn', 'Ali', 'Alic', 'Val']))
    ]
elif modo == 'Costa':
    mi_seleccion = df_master[
        (df_master['City'] == 'Costa') | (df_master['Zone'] == 'Costa')
    ]

print(f"Propiedades en mi_seleccion: {len(mi_seleccion)}")
display(mi_seleccion.head())
print(f"Descripcion de mi_seleccion:\n")
display(mi_seleccion.describe().round(1))


Propiedades en mi_seleccion: 81


,Link,Price (€),Area (m²),Hab,Location,Price/SqM,Type,City,Zone,Location_Full
431,https://www.idealista.com/obra-nueva/105186737/inmueble/105189869/,238000,91.0,2.0,"Centro, Santa Pola",2615.384615,New,Costa,Afueras,Costa Afueras
432,https://www.idealista.com/obra-nueva/105186737/inmueble/105189812/,240000,75.0,2.0,"Centro, Santa Pola",3200.000000,New,Costa,Afueras,Costa Afueras
433,https://www.idealista.com/obra-nueva/105186737/inmueble/105189868/,260000,75.0,2.0,"Centro, Santa Pola",3466.666667,New,Costa,Afueras,Costa Afueras
434,https://www.idealista.com/obra-nueva/105186737/inmueble/105189811/,268000,98.0,2.0,"Centro, Santa Pola",2734.693878,New,Costa,Afueras,Costa Afueras
435,https://www.idealista.com/obra-nueva/105186737/inmueble/105189848/,288000,99.0,3.0,"Centro, Santa Pola",2909.090909,New,Costa,Afueras,Costa Afueras


Descripcion de mi_seleccion:



,Price (€),Area (m²),Hab,Price/SqM
count,81.0,81.0,81.0,81.0
mean,275804.8,85.4,2.3,3305.4
std,27593.0,16.7,0.4,502.8
min,195000.0,65.0,2.0,1932.5
25%,256463.0,74.0,2.0,3070.6
50%,275000.0,81.0,2.0,3355.3
75%,298600.0,95.0,3.0,3625.0
max,320000.0,163.0,3.0,4833.3


### Filtro Estricto de Habitabilidad
Aplica la función avanzada para descartar propiedades con distribuciones extrañas (ej. muchas habitaciones en pocos metros cuadrados).

In [39]:
def filter_buy_rows(df, city=None, zone=None, min_h=2, min_price_sqm=1500, max_price_sqm=3000, min_area_any=60, min_area_large=100, min_hab_large=3, max_size=200, exclude=None):
    cond = df['Price/SqM'].between(min_price_sqm, max_price_sqm) & df['Area (m²)'].between(min_area_any, max_size) & (df['Hab'] >= min_h)
    cond &= (df['Hab'] < min_hab_large) | (df['Area (m²)'] >= min_area_large)
    res = df[cond]
    if city: res = res[res['City'].isin(city)]
    if zone: res = res[res['Zone'].isin(zone)]
    if exclude: res = res[~res['Location'].isin(exclude)]
    return res.drop_duplicates().reset_index(drop=True).sort_values(by="Price/SqM")

propiedades_finales=filter_buy_rows(mi_seleccion,city=None,zone=None,min_h=2,min_price_sqm=3500,max_price_sqm=4500,min_area_any=60,min_area_large=80,min_hab_large=3,max_size=80)

conteo_grupos = propiedades_finales.groupby(['Type', 'City', 'Zone']).size().reset_index(name='Count')
display(conteo_grupos)
pd.set_option('display.max_colwidth', None)
print(f"--- PROPIEDADES ENCONTRADAS TRAS EL FILTRO ESTRICTO> {modo}---\nTotal aptas: {len(propiedades_finales)}")
display(propiedades_finales)


,Type,City,Zone,Count
0,New,Costa,Afueras,15


--- PROPIEDADES ENCONTRADAS TRAS EL FILTRO ESTRICTO> Costa---
Total aptas: 15


,Link,Price (€),Area (m²),Hab,Location,Price/SqM,Type,City,Zone,Location_Full
0,https://www.idealista.com/obra-nueva/110765072/inmueble/110829044/,260000,74.0,2.0,"Centro, Santa Pola",3513.513514,New,Costa,Afueras,Costa Afueras
8,https://www.idealista.com/obra-nueva/107005180/inmueble/107008729/,230000,65.0,2.0,"Les Roquetes, Sant Pere de Ribes",3538.461538,New,Costa,Afueras,Costa Afueras
1,https://www.idealista.com/obra-nueva/110829569/inmueble/110837001/,275000,77.0,2.0,"Centro, Santa Pola",3571.428571,New,Costa,Afueras,Costa Afueras
4,https://www.idealista.com/obra-nueva/107950886/inmueble/111770080/,290000,80.0,2.0,"Playa de Puçol, Puçol",3625.000000,New,Costa,Afueras,Costa Afueras
7,https://www.idealista.com/obra-nueva/110358524/inmueble/110382988/,270000,74.0,2.0,"Les Roquetes, Sant Pere de Ribes",3648.648649,New,Costa,Afueras,Costa Afueras
2,https://www.idealista.com/obra-nueva/107950886/inmueble/111770089/,270000,74.0,2.0,"Playa de Puçol, Puçol",3648.648649,New,Costa,Afueras,Costa Afueras
9,https://www.idealista.com/obra-nueva/107005180/inmueble/107008915/,240000,65.0,2.0,"Les Roquetes, Sant Pere de Ribes",3692.307692,New,Costa,Afueras,Costa Afueras
5,https://www.idealista.com/obra-nueva/107561330/inmueble/110886367/,286190,76.0,2.0,"Playa de Puçol, Puçol",3765.657895,New,Costa,Afueras,Costa Afueras
3,https://www.idealista.com/obra-nueva/107950886/inmueble/111770098/,280000,72.0,2.0,"Playa de Puçol, Puçol",3888.888889,New,Costa,Afueras,Costa Afueras
6,https://www.idealista.com/obra-nueva/107561330/inmueble/110881057/,298600,75.0,2.0,"Playa de Puçol, Puçol",3981.333333,New,Costa,Afueras,Costa Afueras


In [40]:

if 'Link' in propiedades_finales.columns:
    print("\n--- Unidades disponibles por Promoción ---")
    # Recorta el link hasta el ID del proyecto (antes de /inmueble/)
    promo_links = propiedades_finales['Link'].dropna().apply(lambda x: x.split('/inmueble/')[0] + '/' if '/inmueble/' in x else x)
    conteo = promo_links.value_counts()
    print(conteo)
    print("\n")
    
# Abrir todos los links filtrados
links = propiedades_finales['Link'].dropna().tolist()
print(f"Se van a abrir {len(links)} links en tu navegador...")

for link in links:
    webbrowser.open(link)
    time.sleep(0.5)  # Pausa de medio segundo entre cada pestaña


--- Unidades disponibles por Promoción ---
Link
https://www.idealista.com/obra-nueva/107950886/    3
https://www.idealista.com/obra-nueva/109791653/    2
https://www.idealista.com/obra-nueva/107005180/    2
https://www.idealista.com/obra-nueva/110626969/    2
https://www.idealista.com/obra-nueva/107561330/    2
https://www.idealista.com/obra-nueva/110765072/    1
https://www.idealista.com/obra-nueva/110829569/    1
https://www.idealista.com/obra-nueva/110358524/    1
https://www.idealista.com/obra-nueva/109832638/    1
Name: count, dtype: int64


Se van a abrir 15 links en tu navegador...
